# 2016A 系泊系统：解析—数值双重复现

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "请从仓库内运行 Notebook"


## 1. 官方参数

本题无独立数据附件；参数逐项来自官方题面。

In [ ]:
from cumcm_lens.cases.mooring import CHAIN_TYPES, MooringConfig
pd.DataFrame(CHAIN_TYPES).T


## 2. 基线场景

In [ ]:
from cumcm_lens.cases.mooring import solve_state
states = [solve_state(MooringConfig(wind_ms=v)) for v in (12, 24, 36)]
pd.DataFrame([{k:v for k,v in state.items() if not isinstance(v,(list,dict))} for state in states])


## 3. 解析悬链线与数值积分复核

In [ ]:
from cumcm_lens.cases.mooring import numerical_catenary_audit
pd.DataFrame([numerical_catenary_audit(state) for state in states])


## 4. 约束审计

In [ ]:
pd.DataFrame(states[-1]['audits'])

## 5. 鲁棒离散设计搜索

In [ ]:
from cumcm_lens.cases.mooring import design_grid
grid = design_grid()
grid.head(10)


## 6. 敏感性分析

In [ ]:
from cumcm_lens.cases.mooring import sensitivity_table
best = grid[grid["feasible"]].iloc[0]
config = MooringConfig(depth_m=20, wind_ms=36, current_ms=1.5,
    chain_type=best.chain_type, chain_length_m=best.chain_length_m,
    ballast_mass_kg=best.ballast_mass_kg)
sensitivity = sensitivity_table(config)
for name, group in sensitivity.groupby("parameter"):
    plt.plot(group["factor"], group["barrel_angle_deg"], marker="o", label=name)
plt.axhline(5, linestyle="--"); plt.legend(); plt.show()


## 7. 结论边界

模型是稳态、同向风流的静力模型，未覆盖波浪、疲劳、锚土相互作用与制造公差；工程部署前必须进一步验证。